# RAG-Anything — Ingestion sur Colab (GPU)

Ce notebook :
1. Clone le repo + installe les dépendances
2. Installe Ollama sur Colab (GPU gratuit)
3. Ingère les PDFs avec Docling (tableaux complexes)
4. Télécharge le Knowledge Graph généré (`rag_storage.zip`)

## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Cloner le repo et installer les dépendances

In [ ]:
!git clone https://github.com/abdelmajid-byte/rag-anything-project.git
%cd rag-anything-project

In [ ]:
!pip install -q raganything[all] lightrag-hku docling
!pip install -q python-dotenv pyyaml loguru requests numpy fastapi

## 2. Installer et démarrer Ollama avec GPU

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Démarrer Ollama en arrière-plan
import subprocess
import time
import os

# S'assurer que ollama est dans le PATH
os.environ["PATH"] += os.pathsep + "/usr/local/bin"

ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

# Vérifier qu'il tourne
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print(f"Ollama démarré ✅ (status {r.status_code})")
except:
    print("⚠️ Ollama pas encore prêt, attendez quelques secondes et relancez cette cellule")

In [ ]:
# Télécharger les modèles nécessaires
# Adaptez ces noms si vous utilisez d'autres modèles
!ollama pull llama3.1:8b
!ollama pull nomic-embed-text
!ollama pull llava:7b

# Si vous préférez des modèles plus performants (plus lents à télécharger) :
# !ollama pull mistral-nemo:12b
# !ollama pull qwen2.5:7b

## 3. Vérifier que les PDFs sont présents

In [ ]:
import os

# Adapter le chemin si vos PDFs sont dans un dossier différent
DATA_DIR = "./donnees rag"

if not os.path.isdir(DATA_DIR):
    # Si le dossier n'existe pas dans le repo, le créer
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"⚠️  Dossier '{DATA_DIR}' créé. Uploadez vos PDFs dedans (cellule suivante).")
else:
    pdfs = [f for f in os.listdir(DATA_DIR) if f.endswith(".pdf")]
    print(f"📄 {len(pdfs)} PDFs trouvés : {pdfs}")

In [ ]:
# Si les PDFs ne sont pas dans le repo, uploadez-les manuellement :
from google.colab import files

print("Uploadez vos PDFs (ignorez cette cellule si les PDFs sont déjà dans le repo)")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = os.path.join(DATA_DIR, filename)
    with open(dest, "wb") as f:
        f.write(content)
    print(f"  → {filename} enregistré dans {DATA_DIR}")

## 4. Configurer le pipeline pour Colab (GPU + Docling)

In [ ]:
import os

# Forcer Docling + auto (tableaux) + GPU
os.environ["PARSER"] = "docling"
os.environ["PARSE_METHOD"] = "auto"
os.environ["MINERU_DEVICE"] = "cuda"
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

# Modèles — doivent correspondre aux modèles téléchargés (cellule ollama pull)
os.environ["OLLAMA_LLM_MODEL"] = "llama3.1:8b"
os.environ["OLLAMA_EXTRACT_MODEL"] = "llama3.1:8b"
os.environ["OLLAMA_VISION_MODEL"] = "llava:7b"
os.environ["OLLAMA_EMBED_MODEL"] = "nomic-embed-text"

os.environ["HF_ENDPOINT"] = "https://huggingface.co"  # Colab a accès direct à HF

print("Configuration Colab:")
print(f"  Parser      : {os.environ['PARSER']}")
print(f"  Parse method: {os.environ['PARSE_METHOD']}")
print(f"  Device      : {os.environ['MINERU_DEVICE']}")
print(f"  LLM         : {os.environ['OLLAMA_LLM_MODEL']}")
print(f"  Extract     : {os.environ['OLLAMA_EXTRACT_MODEL']}")
print(f"  Vision      : {os.environ['OLLAMA_VISION_MODEL']}")
print(f"  Embedding   : {os.environ['OLLAMA_EMBED_MODEL']}")

## 5. Lancer l'ingestion

In [ ]:
import sys
sys.path.insert(0, "src")

import asyncio
from ingestion.rag_anything_pipeline import ingest_all_documents, check_ollama_models

# Vérifier Ollama
status = check_ollama_models()
for role, info in status.items():
    icon = "✅" if info.get("available") else "❌"
    print(f"  {icon} {role}: {info['model']}")

all_ok = all(info.get("available") for info in status.values())
if not all_ok:
    raise RuntimeError("Modèles Ollama manquants ! Relancez la cellule 'ollama pull'.")

print("\n🚀 Démarrage de l'ingestion...")

In [ ]:
# Supprimer l'ancien index si présent
import shutil

WORKING_DIR = "./data/rag_anything_storage"
if os.path.exists(WORKING_DIR):
    shutil.rmtree(WORKING_DIR)
    print(f"Ancien index supprimé: {WORKING_DIR}")

# Lancer l'ingestion
rag = await ingest_all_documents(
    data_dir=DATA_DIR,
    output_dir="./output",
    working_dir=WORKING_DIR,
)
print("\n✅ Ingestion terminée !")

## 6. Test rapide — vérifier la qualité

In [ ]:
from ingestion.rag_anything_pipeline import query

questions = [
    "Quels sont les trois formats obligatoires pour transmettre les factures électroniques ?",
    "Quelle est la date d'entrée en vigueur de l'obligation de facturation électronique ?",
    "Qu'est-ce qu'une plateforme de dématérialisation partenaire (PDP) ?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    answer = await query(q, mode="hybrid")
    print(f"R: {answer}")

## 7. Télécharger le Knowledge Graph

Le dossier `data/rag_anything_storage/` contient le Knowledge Graph complet.
Téléchargez-le pour l'utiliser en local.

In [ ]:
# Compresser le Knowledge Graph
!zip -r rag_storage.zip data/rag_anything_storage/

# Télécharger
from google.colab import files
files.download("rag_storage.zip")

print("\n📦 Téléchargez rag_storage.zip puis décompressez-le dans votre projet local :")
print("   unzip rag_storage.zip -d 'C:/Users/.../RAG benchmark_RAG-AnythingVV4/'")

## 8. (Optionnel) Pousser le storage sur le repo Git

In [ ]:
# Décommentez et adaptez si vous voulez push le storage sur GitHub
# !git add data/rag_anything_storage/
# !git commit -m "Add ingested knowledge graph (Docling + GPU)"
# !git push origin main